In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import Sequential,layers
import zipfile

In [2]:
zip =zipfile.ZipFile("/content/drive/MyDrive/Fruits_Vegetables.zip")

In [3]:
zip.extractall("Fruits_and_vegetable")

In [4]:
image_height=180
image_width=180

In [5]:
train_images=tf.keras.utils.image_dataset_from_directory(
    "/content/Fruits_and_vegetable/Fruits_Vegetables/train",
    image_size=(image_height,image_width),
  batch_size=32,
    shuffle=True,
    seed=42,

    validation_split=False
)

Found 3115 files belonging to 36 classes.


In [6]:
data_label=train_images.class_names


In [7]:
test_images=tf.keras.utils.image_dataset_from_directory(
    "/content/Fruits_and_vegetable/Fruits_Vegetables/test",
    image_size=(image_height,image_width),
  batch_size=32,
    shuffle=True,
    seed=42,
    validation_split=False
)

Found 359 files belonging to 36 classes.


In [8]:
val_split = tf.keras.utils.image_dataset_from_directory(
    "/content/Fruits_and_vegetable/Fruits_Vegetables/validation",
    image_size=(image_height, image_width),
    batch_size=32,
    shuffle=True,
    seed=42
)

Found 351 files belonging to 36 classes.


In [9]:
model=Sequential([
    layers.Rescaling(1./255),
    layers.Conv2D(16,3,padding="same",activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(32,3,padding="same",activation="relu"),
    layers.MaxPooling2D(2),
    layers.Conv2D(64,3,padding="same",activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(128,activation="relu"),
    layers.Dense(len(data_label))


])

In [10]:
model.compile(optimizer='adam',loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),metrics=['accuracy'])

In [11]:
history = model.fit(
    train_images,
    validation_data=val_split,
    epochs=5,
    batch_size=32
)

Epoch 1/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 200s 2s/step - accuracy: 0.0982 - loss: 3.2927 - val_accuracy: 0.3048 - val_loss: 2.5481
Epoch 2/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 193s 2s/step - accuracy: 0.2758 - loss: 2.5433 - val_accuracy: 0.4786 - val_loss: 1.8074
Epoch 3/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 208s 2s/step - accuracy: 0.4270 - loss: 1.9472 - val_accuracy: 0.6724 - val_loss: 1.1950
Epoch 4/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 195s 2s/step - accuracy: 0.5910 - loss: 1.3930 - val_accuracy: 0.8348 - val_loss: 0.7195
Epoch 5/5
98/98 ━━━━━━━━━━━━━━━━━━━━ 199s 2s/step - accuracy: 0.7576 - loss: 0.8225 - val_accuracy: 0.8974 - val_loss: 0.4584


In [13]:
import PIL
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

In [17]:
def predictor(image):

    img = tf.keras.utils.load_img(image, target_size=(image_height, image_width))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, 0)

    predictions = model.predict(img_array)
    score = tf.nn.softmax(predictions[0])

    confidence = np.max(score) * 100
    predicted_label = data_label[np.argmax(score)]

    return f"The provided image is a {predicted_label}, and model confidence is {confidence:.2f}%"